In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [3]:
df.dtypes

Unnamed: 0             int64
key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

In [4]:
df = df.drop(columns=['Unnamed: 0', 'key'], axis=1)
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [5]:
df.isna().sum()


fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64

In [6]:
df.fillna(df['dropoff_latitude'].mean(), inplace=True)
df.fillna(df['dropoff_longitude'].mean(), inplace=True)
df.isna().sum()
df

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5
...,...,...,...,...,...,...,...
199995,3.0,2012-10-28 10:49:00 UTC,-73.987042,40.739367,-73.986525,40.740297,1
199996,7.5,2014-03-14 01:09:00 UTC,-73.984722,40.736837,-74.006672,40.739620,1
199997,30.9,2009-06-29 00:42:00 UTC,-73.986017,40.756487,-73.858957,40.692588,2
199998,14.5,2015-05-20 14:56:25 UTC,-73.997124,40.725452,-73.983215,40.695415,1


In [7]:
df = df[df['fare_amount']>=0]

In [8]:
df.corr(method="pearson", numeric_only=True)

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
fare_amount,1.000000,0.010432,-0.008460,0.009198,-0.011183,0.010403
pickup_longitude,0.010432,1.000000,-0.816461,0.832935,-0.846401,-0.000411
pickup_latitude,-0.008460,-0.816461,1.000000,-0.774702,0.702430,-0.001563
dropoff_longitude,0.009198,0.832935,-0.774702,1.000000,-0.916829,-0.000083
dropoff_latitude,-0.011183,-0.846401,0.702430,-0.916829,1.000000,-0.000591
passenger_count,0.010403,-0.000411,-0.001563,-0.000083,-0.000591,1.000000


In [9]:
def remove_outliers(feature):
    global df
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5*iqr
    upper_bound = q3 + 1.5*iqr
    df = df[(df[feature]>=lower_bound) & (df[feature]<=upper_bound)]

remove_outliers("pickup_latitude")
remove_outliers("pickup_longitude")
remove_outliers("dropoff_latitude")
remove_outliers("dropoff_longitude")

df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,171169.000000,171169.000000,171169.000000,171169.000000,171169.000000,171169.000000
mean,9.012359,-73.981929,40.752519,-73.980935,40.752778,1.683623
std,4.939390,0.016110,0.021653,0.016904,0.022960,1.399014
min,0.000000,-74.026353,40.686252,-74.027438,40.687874,0.000000
25%,5.700000,-73.992917,40.737725,-73.992299,40.737588,1.000000
50%,8.000000,-73.982912,40.753372,-73.982013,40.753978,1.000000
75%,11.000000,-73.971443,40.766691,-73.970250,40.767489,2.000000
max,499.000000,-73.935240,40.815686,-73.932840,40.816622,208.000000


In [10]:
import math

def euclidean_distance(lat1, lon1, lat2, lon2):
    dlat = lat2-lat1
    dlon = lon2-lon1
    distance = math.sqrt(dlat**2 + dlon**2)
    return distance

df['distance'] = df.apply(
    lambda row: euclidean_distance(row['pickup_latitude'], row['pickup_longitude'], row['dropoff_latitude'],row['dropoff_longitude']
                                   ), axis = 1
)

In [ ]:
df.drop(['pickup_latitude', 'pickup_longitude','dropoff_latitude', 'dropoff_longitude'], axis=1, inplace=True)

df.head()


In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], utc=True)
df['hour'] = df['pickup_datetime'].dt.hour

df.drop(['pickup_datetime'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.corr(method="pearson", numeric_only=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('fare_amount', axis = 1)
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled , y_train)

y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

print("R2 score: ", r2_score(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=10, max_depth=5)
model.fit(X_train_scaled, y_train)



In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

print("R2 score: ", r2_score(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))